# 서울 사고누적지점의 사고 유형별 지도
2023~2025년 사망·중상사고 자료에서 대분류 또는 세부 유형을 선택해 지점별 사고 비중을 확인한다. 원의 크기는 해당 유형의 비중을 나타낸다. 분모는 그 지점의 전체 사고 수다.

In [1]:
import csv
import json
from collections import Counter
from pathlib import Path
from html import escape
from IPython.display import HTML, display

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/processed/사고_분석용.csv").is_file())
def read_csv(path):
    with path.open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

rows = read_csv(root / "data/processed/사고_분석용.csv")
sites = read_csv(root / "data/raw/tmacs/2025/지점목록.csv")
print(f"분석 기간 2023~2025년 · {len(rows):,}건")

분석 기간 2023~2025년 · 4,564건


## 지점별 건수와 비중 집계
분석용 CSV에서 자치구·지점명·중심 좌표를 사용해 지점목록의 지점ID를 찾는다. 각 지점의 집계 건수가 원본 목록과 일치하는지 확인한다.

In [2]:
def site_key(row):
    return tuple(row[k] for k in ("자치구", "지점명", "지점위도", "지점경도"))

points = {site_key(s): dict(id=s["지점ID"], name=s["지점명"], gu=s["자치구"],
          lat=float(s["지점위도"]), lon=float(s["지점경도"]), total=0,
          major=Counter(), detail=Counter()) for s in sites}
assert len(points) == len(sites)
for row in rows:
    point = points[site_key(row)]
    point["total"] += 1
    point["major"][row["사고유형_대분류"] or "미상"] += 1
    point["detail"][row["사고유형_세부"] or "미상"] += 1
assert all(points[site_key(s)]["total"] == int(s["지점사고건수"]) for s in sites)
print(f"지점 {len(points)}곳 검증 완료")
for label, count in Counter(r["사고유형_대분류"] for r in rows).most_common():
    print(f"{label}: {count:,}건 ({count / len(rows):.1%})")

지점 327곳 검증 완료
차대차: 3,118건 (68.3%)
차대사람: 1,203건 (26.4%)
차량단독: 243건 (5.3%)


## 지도에서 유형 비교
분류와 유형을 고르면 해당 사고가 발생한 지점만 표시된다. 원을 클릭하면 건수와 분모를 함께 볼 수 있다. 지도 배경과 Leaflet을 불러오려면 인터넷 연결이 필요하다.

In [3]:
page = """<!doctype html><html lang="ko"><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>서울 사고 유형 지도</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<style>body{margin:0;font:16px system-ui;color:#162d41}header{padding:12px 18px}h2{margin:0 0 8px}select{padding:4px;margin-right:10px}#map{height:72vh;min-height:320px}p{margin:8px 0}</style>
<header><h2>서울 사고누적지점 · 2023~2025</h2><label>분류 <select id="level"><option value="major">대분류</option><option value="detail">세부 유형</option></select></label>
<label>사고 유형 <select id="type"></select></label><p id="status"></p><small>원 면적은 각 지점에서 선택 유형이 차지하는 비중에 비례합니다.</small></header><div id="map"></div>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script><script>
const points = __POINTS__;
const map = L.map("map").fitBounds(points.map(p=>[p.lat,p.lon]));
L.tileLayer("https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",{attribution:'&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors'}).addTo(map);
const layer=L.layerGroup().addTo(map), level=document.querySelector("#level"), type=document.querySelector("#type");
function draw(){
 layer.clearLayers(); let sites=0, cases=0;
 for(const p of points){
  const n=p[level.value][type.value] || 0;
  if(!n) continue;
  sites++; cases+=n;
  const share=n/p.total;
  const label=document.createElement("div");
  label.textContent=p.gu+" "+p.name+" · "+type.value+" "+n+"건 / 전체 "+p.total+"건 ("+(100*share).toFixed(1)+"%)";
  L.circleMarker([p.lat,p.lon],{radius:19*Math.sqrt(share),color:"#124865",weight:1,fillColor:"#008c97",fillOpacity:0.62}).bindPopup(label).addTo(layer);
 }
 document.querySelector("#status").textContent=type.value+" · "+sites+"개 지점 · "+cases.toLocaleString()+"건";
}
function options(){
 const totals={};
 for(const p of points) for(const [k,n] of Object.entries(p[level.value])) totals[k]=(totals[k]||0)+n;
 type.replaceChildren(...Object.keys(totals).sort((a,b)=>totals[b]-totals[a]).map(k=>new Option(k,k)));
 draw();
}
level.onchange=options; type.onchange=draw; options();
</script></html>"""
payload = json.dumps(list(points.values()), ensure_ascii=False).replace("<", r"\u003c")
page = page.replace("__POINTS__", payload)
output = root / "outputs/서울_사고유형_지도.html"
output.parent.mkdir(exist_ok=True)
output.write_text(page, encoding="utf-8")
print(f"지도 저장: {output.name}")
display(HTML('<iframe title="서울 사고 유형 지도" style="width:100%;height:650px;border:0" srcdoc="' + escape(page, quote=True) + '"></iframe>'))

지도 저장: 서울_사고유형_지도.html


C:\Python\Python312\Lib\site-packages\IPython\core\display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")
